In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy -q
print("Done!")

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset
import pandas as pd
import gc

print("Downloading dataset...")

ds = load_dataset("artem9k/ai-text-detection-pile", split="train")
df = ds.to_pandas()

del ds
gc.collect()

print(df.shape)
print(df['source'].value_counts())

In [ ]:
human_df = df[df['source'] == 'human'][['text']].copy()
human_df['label'] = 0

ai_df = df[df['source'] == 'ai'][['text']].copy()
ai_df['label'] = 1

n = min(25000, len(human_df), len(ai_df))

balanced = pd.concat([
    human_df.sample(n, random_state=42),
    ai_df.sample(n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

balanced = balanced[['text', 'label']].dropna()
balanced['text'] = balanced['text'].astype(str)

del df, human_df, ai_df
gc.collect()

print(balanced.shape)
print(balanced['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    balanced,
    test_size=0.2,
    random_state=42,
    stratify=balanced['label']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df['text'])
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_df['text'])

print("Tokenization completed!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TextDataset(train_encodings, train_df['label'])
val_dataset = TextDataset(val_encodings, val_df['label'])
test_dataset = TextDataset(test_encodings, test_df['label'])

print("Dataset created!")

In [ ]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

print("RoBERTa model loaded!")

In [ ]:
print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("Kernel is working")

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated()/1024**3, "GB")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model moved to:", device)
print(torch.cuda.memory_allocated()/1024**3, "GB")

In [ ]:
from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    logging_first_step=True,
    disable_tqdm=False,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Training started...")
trainer.train()
print("Training completed!")

In [ ]:
print("kernel working")

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

test_loader = DataLoader(test_dataset, batch_size=32)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = labels.to(device)

        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Human", "AI"]))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
model_path = "/kaggle/working/roberta_ai_detector"

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Model saved successfully!")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

def predict_text(text):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    label = "👤 Human Written" if pred == 0 else "🤖 AI Generated"

    print(f"\nPrediction: {label}")
    print(f"Confidence: {confidence*100:.2f}%")

while True:
    text = input("\nEnter text (or type quit): ")

    if text.lower() == "quit":
        break

    predict_text(text)